# CRAFT Stage 2: Dense Semantic Reranking

Takes Stage 1 SPLADE candidates (5,000 per query) and reranks them using dense embeddings over **mini-tables** (top-5 most relevant rows per table). Outputs top-100 candidates for Stage 3.

**Two modes:**
- `USE_PRECOMPUTED = True` (default): load already-computed rankings from disk — fast, no GPU needed.
- `USE_PRECOMPUTED = False`: build mini-tables, encode with a Sentence Transformer, and rerank from scratch.

**Datasets:**
- `nq`: all-mpnet-base-v2 embeddings
- `ottqa`: JINA Embeddings v3

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
os.environ['HF_HOME'] = '/mnt/data2/asing725_2/hf_cache'
# Set HF_TOKEN in your environment before running: export HF_TOKEN=hf_...

CACHE_DIR = '/mnt/data2/asing725_2/hf_cache'

In [17]:
import sys
import json
import pickle
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
from tqdm.notebook import tqdm

# Resolve repo root the same way stage1 does
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'utils').exists():
    if (REPO_ROOT / 'CRAFT' / 'utils').exists():
        REPO_ROOT = REPO_ROOT / 'CRAFT'
    else:
        REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from utils.io_utils import load_pickle, save_pickle, read_jsonl, write_jsonl
from utils.eval_metric import evaluate_recall

DATA_DIR    = REPO_ROOT / 'datasets'
RESULTS_DIR = REPO_ROOT / 'results' / 'stage2'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Repo root:', REPO_ROOT)

Device: cuda
Repo root: /mnt/data1/asing725/ACL/CRAFT


## Configuration

Change `DATASET` and `USE_PRECOMPUTED` here — everything else is derived automatically.

In [18]:
# ── Main switches ──────────────────────────────────────────────────────────────
DATASET         = 'nq'   # 'nq' or 'ottqa'
USE_PRECOMPUTED = True   # True → load saved rankings; False → recompute from scratch

# ── Pipeline constants ─────────────────────────────────────────────────────────
TOP_K_ROWS   = 5     # rows per table used to build each mini-table
STAGE2_TOP_K = 100   # tables kept per query → passed to Stage 3
EMBED_BATCH  = 256   # batch size for sentence-transformer encoding

RECALL_KS = [1, 10, 50, 100, 500]

# ── Dataset-specific paths ─────────────────────────────────────────────────────
if DATASET == 'nq':
    STAGE1_PATH      = REPO_ROOT / 'results' / 'stage1' / 'nq_stage1.jsonl'
    QUESTIONS_PATH   = None   # NQ questions are embedded in the stage1 file
    ROW_RERANK_PATH  = DATA_DIR / 'corpus_2nd_stage_row_rerank_with_ST_original_q.pkl'
    ROW_TABLES_PATH  = DATA_DIR / 'nq_row_tables_INDIV_Table.json'
    PRECOMPUTED_PATH = DATA_DIR / 'corpus_top5000_reranked_with_SentTrasf_title_head_desc.pkl'
    EMBED_MODEL_ID   = 'sentence-transformers/all-mpnet-base-v2'
    OUTPUT_PATH      = RESULTS_DIR / 'nq_stage2.jsonl'

elif DATASET == 'ottqa':
    # Separate questions file (has question_id, question description, gold table)
    QUESTIONS_PATH   = Path('/mnt/data1/asing725/ottqa/data/OTT_QA_Qeuery_Desc.jsonl')
    # Stage 1 SPLADE results (different field names from NQ — normalised in Step 1)
    STAGE1_PATH      = Path('/mnt/data1/asing725/ottqa/data/OTT_QA_STAGE1_RESULTS.jsonl')
    ROW_RERANK_PATH  = DATA_DIR / 'ottqa_top_rows.pkl'
    ROW_TABLES_PATH  = DATA_DIR / 'ottqa_row_tables.pkl'
    PRECOMPUTED_PATH = DATA_DIR / 'ottqa_stage2_results.jsonl'
    EMBED_MODEL_ID   = 'jinaai/jina-embeddings-v3'
    OUTPUT_PATH      = RESULTS_DIR / 'ottqa_stage2.jsonl'

else:
    raise ValueError(f'Unknown dataset: {DATASET!r}. Use "nq" or "ottqa".')

print(f'Dataset        : {DATASET}')
print(f'Use precomputed: {USE_PRECOMPUTED}')
print(f'Output path    : {OUTPUT_PATH}')

Dataset        : nq
Use precomputed: True
Output path    : /mnt/data1/asing725/ACL/CRAFT/results/stage2/nq_stage2.jsonl


## Step 1 — Load & Normalise Stage 1 Results

Both datasets are normalised to the same internal format:
```json
{"qid": "...", "question": "...", "gold_table_ids": ["..."], "retrieved": [{"rank": 1, "table_id": "...", "score": ...}]}
```

**NQ** — already in this format from stage1.ipynb.  
**OTT-QA** — raw fields: `Qid` (int), `Query`, `Gold Table ID`, `Topk_TableIDs` (`[[id, score], ...]`).  
We match `Query` text → `question_id` (hex) using the separate questions file.

In [19]:
if DATASET == 'nq':
    stage1_results = list(read_jsonl(STAGE1_PATH))

elif DATASET == 'ottqa':
    # ── Load questions file: question_id (hex) + question text ──────────────
    questions_raw = list(read_jsonl(QUESTIONS_PATH))
    # Map question text → question metadata for ID lookup
    q_text_to_meta = {item['question'].strip(): item for item in questions_raw}
    print(f'Questions file  : {len(questions_raw)} entries')

    # ── Load & normalise Stage 1 ─────────────────────────────────────────────
    raw_s1 = list(read_jsonl(STAGE1_PATH))
    stage1_results = []
    skipped = 0
    for item in raw_s1:
        qtext = item['Query'].strip()
        meta  = q_text_to_meta.get(qtext)
        if meta is None:
            skipped += 1
            continue
        retrieved = [
            {'rank': i + 1, 'table_id': tid, 'score': float(score)}
            for i, (tid, score) in enumerate(item['Topk_TableIDs'])
        ]
        stage1_results.append({
            'qid'           : meta['question_id'],
            'question'      : qtext,
            'gold_table_ids': [item['Gold Table ID']],
            'retrieved'     : retrieved,
        })
    print(f'Stage 1 loaded  : {len(stage1_results)} (skipped {skipped} unmatched)')

# Build a quick lookup: qid → stage1 item (used later for gold_table_ids)
stage1_by_qid = {item['qid']: item for item in stage1_results}

print(f'Stage 1 results : {len(stage1_results)} questions')
ex = stage1_results[0]
print(f'Example qid     : {ex["qid"]}')
print(f'Question        : {ex["question"][:70]}')
print(f'Gold table IDs  : {ex["gold_table_ids"]}')
print(f'Candidates      : {len(ex["retrieved"])} tables')
print(f'Top-3 tables    : {[r["table_id"] for r in ex["retrieved"][:3]]}')

Stage 1 results : 966 questions
Example qid     : dev_6330519627947400943_0
Question        : where does the brazos river start and stop
Gold table IDs  : ['Brazos River_8F7B4BA175AC5E8F']
Candidates      : 5000 tables
Top-3 tables    : ['Brazos River_8F7B4BA175AC5E8F', 'Battle of the Brazos_21AE7A045C15948C', 'Brahmaputra River_7B89293C11AF23AD']


## Step 2 — Build Row-Data Index

We need to look up a row by *(table_id, row_number)* in O(1).

- **NQ** — `nq_row_tables_INDIV_Table.json`: flat list; key fields `Table Id`, `table_row_number`, `Row Data`.
- **OTT-QA** — `ottqa_row_tables.pkl`: flat list; rows accessed by absolute list index.

This step is skipped when `USE_PRECOMPUTED = True` since we never build mini-tables.

In [20]:
row_index = None

if not USE_PRECOMPUTED:
    print('Building row-data index …')

    if DATASET == 'nq':
        with open(ROW_TABLES_PATH, 'r') as f:
            raw_rows = json.load(f)
        # row_index[table_id][table_row_number] = row_text
        row_index = defaultdict(dict)
        for row in raw_rows:
            row_index[row['Table Id']][row['table_row_number']] = row.get('Row Data', '')
        print(f'  NQ row index: {len(row_index):,} tables, {len(raw_rows):,} total rows')

    elif DATASET == 'ottqa':
        raw_rows = load_pickle(ROW_TABLES_PATH)
        # ottqa_top_rows gives absolute list indices → keep as flat list
        row_index = raw_rows
        print(f'  OTT-QA row list: {len(row_index):,} rows')
else:
    print('Skipping row index (USE_PRECOMPUTED=True)')

Skipping row index (USE_PRECOMPUTED=True)


## Step 3 — Load Row-Rank Data

For each *(question, table)* pair we need the top-K most relevant row indices, pre-ranked by a Sentence Transformer.

- **NQ** `corpus_2nd_stage_row_rerank_with_ST_original_q.pkl`  
  `{ qid → { table_id → [row_numbers_within_table] } }`  (row numbers are `table_row_number`, 0-indexed)

- **OTT-QA** `ottqa_top_rows.pkl`  
  `{ qid → { table_id → [absolute_row_indices] } }`  (direct indices into `ottqa_row_tables` list)

Skipped when `USE_PRECOMPUTED = True`.

In [21]:
row_rerank = None

if not USE_PRECOMPUTED:
    row_rerank = load_pickle(ROW_RERANK_PATH)
    print(f'Row-rerank entries: {len(row_rerank):,} questions')
    sample_qid    = next(iter(row_rerank))
    sample_tables = list(row_rerank[sample_qid].items())[:2]
    for tid, rows in sample_tables:
        print(f'  {tid}: top rows = {rows[:TOP_K_ROWS]}')
else:
    print('Skipping row-rerank load (USE_PRECOMPUTED=True)')

Skipping row-rerank load (USE_PRECOMPUTED=True)


## Step 4 — Mini-table Builder

For each *(question, table)* pair:
1. Look up the top-`TOP_K_ROWS` row indices from `row_rerank`.
2. Fetch each row's text from `row_index`.
3. Concatenate into a single string (the mini-table).

Falls back to the `table_id` string if no row data is found.

In [22]:
def build_minitable_nq(qid: str, table_id: str) -> str:
    """Mini-table for NQ: top-K row texts joined by space."""
    top_row_nums  = row_rerank.get(qid, {}).get(table_id, [])[:TOP_K_ROWS]
    table_row_map = row_index.get(table_id, {})
    texts = [table_row_map[rn].strip() for rn in top_row_nums if table_row_map.get(rn, '').strip()]
    return ' '.join(texts) if texts else table_id.replace('_', ' ')


def build_minitable_ottqa(qid: str, table_id: str) -> str:
    """Mini-table for OTT-QA: top-K absolute-indexed rows joined by space."""
    abs_indices = row_rerank.get(qid, {}).get(table_id, [])[:TOP_K_ROWS]
    texts = []
    for idx in abs_indices:
        if idx < len(row_index):
            row  = row_index[idx]
            text = row.get('Row Data', row.get('row_data', '')).strip()
            if text:
                texts.append(text)
    return ' '.join(texts) if texts else table_id.replace('_', ' ')


build_minitable = build_minitable_nq if DATASET == 'nq' else build_minitable_ottqa

# Sanity check (compute mode only)
if not USE_PRECOMPUTED and row_rerank is not None:
    ex_qid   = stage1_results[0]['qid']
    ex_table = stage1_results[0]['retrieved'][0]['table_id']
    mt = build_minitable(ex_qid, ex_table)
    print('Mini-table preview:')
    print(' ', mt[:300])
else:
    print('Mini-table builder defined (used only when USE_PRECOMPUTED=False)')

Mini-table builder defined (used only when USE_PRECOMPUTED=False)


## Step 5 — Load or Compute Stage 2 Rankings

### 5a. Load pre-computed results (default, `USE_PRECOMPUTED = True`)

| Dataset | File | Format |
|---------|------|--------|
| NQ | `corpus_top5000_reranked_with_SentTrasf_title_head_desc.pkl` | `{'0': [table_id, …], '1': …}` — query-index (str) → ranked table list |
| OTT-QA | `ottqa_stage2_results.jsonl` | `{qid, ranked_tables: [{table_id, score}, …], gold_table_id, …}` |

Both are converted to the unified output format before saving.

In [23]:
stage2_results = []

if USE_PRECOMPUTED:
    print('Loading pre-computed Stage 2 rankings …')

    # ── NQ ────────────────────────────────────────────────────────────────────
    if DATASET == 'nq':
        # PKL indexed by string integer matching the order of stage1_results
        precomp = load_pickle(PRECOMPUTED_PATH)
        print(f'  Loaded rankings for {len(precomp)} queries')

        for idx, item in enumerate(stage1_results):
            ranked_tables = precomp.get(str(idx), [])
            top_tables    = ranked_tables[:STAGE2_TOP_K]
            retrieved     = [{'rank': r + 1, 'table_id': tid} for r, tid in enumerate(top_tables)]
            stage2_results.append({
                'qid'           : item['qid'],
                'question'      : item['question'],
                'gold_table_ids': item['gold_table_ids'],
                'retrieved'     : retrieved,
            })

    # ── OTT-QA ───────────────────────────────────────────────────────────────
    elif DATASET == 'ottqa':
        # JSONL: qid → ranked_tables (dense-reranked order; scores not saved)
        # We restrict to questions that also appear in the Stage 1 file for
        # a fair before/after comparison in the final evaluation cell.
        precomp_list   = list(read_jsonl(PRECOMPUTED_PATH))
        precomp_by_qid = {r['qid']: r for r in precomp_list}
        print(f'  Precomputed file: {len(precomp_by_qid)} queries')

        for item in stage1_results:  # only the 2198 with Stage 1 data
            pc = precomp_by_qid.get(item['qid'], {})
            ranked_tables = pc.get('ranked_tables', [])
            top_tables    = ranked_tables[:STAGE2_TOP_K]
            retrieved     = [
                {'rank': r + 1, 'table_id': t['table_id'],
                 'score': t.get('score')}   # score is None in saved file
                for r, t in enumerate(top_tables)
            ]
            stage2_results.append({
                'qid'           : item['qid'],
                'question'      : item['question'],
                'gold_table_ids': item['gold_table_ids'],
                'retrieved'     : retrieved,
            })

    print(f'Stage 2 assembled: {len(stage2_results)} questions')

Loading pre-computed Stage 2 rankings …
  Loaded rankings for 966 queries
Stage 2 assembled: 966 questions


### 5b. Compute from scratch (`USE_PRECOMPUTED = False`)

For each question:
1. Build mini-table strings for all 5,000 SPLADE candidates.
2. Encode the question and all mini-tables with the Sentence Transformer.
3. Rank tables by cosine similarity (dot product of L2-normalised vectors).

This cell is **skipped** when `USE_PRECOMPUTED = True`.

In [24]:
if not USE_PRECOMPUTED:
    from sentence_transformers import SentenceTransformer

    print(f'Loading embedding model: {EMBED_MODEL_ID}')
    model_kwargs = dict(cache_folder=CACHE_DIR, device=DEVICE)
    if DATASET == 'ottqa':
        model_kwargs['trust_remote_code'] = True   # required by JINA v3
    embed_model = SentenceTransformer(EMBED_MODEL_ID, **model_kwargs)
    print('Model loaded.')


    def encode_texts(texts: list[str], task: str | None = None) -> np.ndarray:
        """Encode texts → L2-normalised float32 matrix (N, D).
        Dot product of two normalised vectors equals cosine similarity.
        """
        kwargs = dict(
            batch_size=EMBED_BATCH,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        if task:
            kwargs['task'] = task  # JINA v3 task prompt
        return embed_model.encode(texts, **kwargs)


    for item in tqdm(stage1_results, desc='Stage 2 reranking'):
        qid        = item['qid']
        question   = item['question']
        candidates = [r['table_id'] for r in item['retrieved']]  # 5000 ids

        # Build mini-table strings for all candidates
        mini_tables = [build_minitable(qid, tid) for tid in candidates]

        # Encode query
        q_task  = 'retrieval.query' if DATASET == 'ottqa' else None
        q_vec   = encode_texts([question], task=q_task)[0]  # shape (D,)

        # Encode mini-tables
        mt_task = 'retrieval.passage' if DATASET == 'ottqa' else None
        mt_vecs = encode_texts(mini_tables, task=mt_task)   # shape (5000, D)

        # Rank by cosine similarity (descending)
        scores     = mt_vecs @ q_vec          # shape (5000,)
        ranked_idx = np.argsort(-scores)       # highest first

        top_idx    = ranked_idx[:STAGE2_TOP_K]
        retrieved  = [
            {'rank': r + 1, 'table_id': candidates[i], 'score': float(scores[i])}
            for r, i in enumerate(top_idx)
        ]
        stage2_results.append({
            'qid'           : qid,
            'question'      : question,
            'gold_table_ids': item['gold_table_ids'],
            'retrieved'     : retrieved,
        })

    print(f'Computed Stage 2 results for {len(stage2_results)} questions')

## Step 6 — Save Results

Output format (same for both datasets):
```json
{"qid": "...", "question": "...", "gold_table_ids": ["..."], "retrieved": [{"rank": 1, "table_id": "..."}]}
```

In [25]:
write_jsonl(OUTPUT_PATH, stage2_results)
print(f'Saved {len(stage2_results)} results → {OUTPUT_PATH}')

ex = stage2_results[0]
print('\nExample output:')
print('  qid           :', ex['qid'])
print('  question      :', ex['question'][:70])
print('  gold_table_ids:', ex['gold_table_ids'])
print(f'  retrieved     : {len(ex["retrieved"])} tables (top-{STAGE2_TOP_K})')
print('  top-3 tables  :', [r['table_id'] for r in ex['retrieved'][:3]])

Saved 966 results → /mnt/data1/asing725/ACL/CRAFT/results/stage2/nq_stage2.jsonl

Example output:
  qid           : dev_6330519627947400943_0
  question      : where does the brazos river start and stop
  gold_table_ids: ['Brazos River_8F7B4BA175AC5E8F']
  retrieved     : 100 tables (top-100)
  top-3 tables  : ['Brazos River_8F7B4BA175AC5E8F', 'Pecos River_2220919271171CD3', 'Amazon River_A2081A95A9890F59']


## Step 7 — Before / After Recall Comparison

Side-by-side Recall@k for **Stage 1 (SPLADE)** vs **Stage 2 (Dense Reranking)** across both datasets.  
Both datasets are evaluated on the **same** set of questions that have Stage 1 results, ensuring a fair comparison.

> **Note on Recall@500:** Stage 2 outputs only the top-`STAGE2_TOP_K` (100) tables per query, so Recall@500 = Recall@100 for Stage 2 by construction. The drop vs Stage 1 Recall@500 is expected — Stage 3 receives only these 100 candidates.

In [ ]:
import pandas as pd
from IPython.display import display

EVAL_KS = [1, 10, 50, 100, 500]

def stage1_recall_format(items):
    return [{'qid': it['qid'], 'gold_table_ids': it['gold_table_ids'],
             'retrieved': it['retrieved']} for it in items]

def stage2_recall_format(items):
    return [{'qid': it['qid'], 'gold_table_ids': it['gold_table_ids'],
             'retrieved': it['retrieved']} for it in items]

def pct_improvement(s1, s2):
    """Relative improvement: (s2 - s1) / s1 * 100."""
    if s1 == 0:
        return 'N/A'
    return f"{(s2 - s1) / s1 * 100:+.1f}%"

# ── Current dataset (already in memory) ──────────────────────────────────────
s1_metrics_cur = evaluate_recall(stage1_recall_format(stage1_results), EVAL_KS)
s2_metrics_cur = evaluate_recall(stage2_recall_format(stage2_results), EVAL_KS)

# ── Other dataset (load from disk) ───────────────────────────────────────────
if DATASET == 'nq':
    _q_raw    = list(read_jsonl('/mnt/data1/asing725/ottqa/data/OTT_QA_Qeuery_Desc.jsonl'))
    _qmap     = {it['question'].strip(): it for it in _q_raw}
    _s1_raw   = list(read_jsonl('/mnt/data1/asing725/ottqa/data/OTT_QA_STAGE1_RESULTS.jsonl'))
    _s1_other = []
    for it in _s1_raw:
        meta = _qmap.get(it['Query'].strip())
        if meta is None:
            continue
        _s1_other.append({
            'qid'           : meta['question_id'],
            'gold_table_ids': [it['Gold Table ID']],
            'retrieved'     : [{'rank': i+1, 'table_id': t[0], 'score': float(t[1])}
                               for i, t in enumerate(it['Topk_TableIDs'])],
        })
    _s1_qids  = {it['qid'] for it in _s1_other}
    _s2_raw   = list(read_jsonl(DATA_DIR / 'ottqa_stage2_results.jsonl'))
    _s2_other = []
    for it in _s2_raw:
        if it['qid'] not in _s1_qids:
            continue
        top = it.get('ranked_tables', [])[:STAGE2_TOP_K]
        _s2_other.append({
            'qid'           : it['qid'],
            'gold_table_ids': [it['gold_table_id']],
            'retrieved'     : [{'rank': r+1, 'table_id': t['table_id']} for r, t in enumerate(top)],
        })
    s1_metrics_other = evaluate_recall(_s1_other, EVAL_KS)
    s2_metrics_other = evaluate_recall(_s2_other, EVAL_KS)
    n_cur, n_other   = len(stage1_results), len(_s1_other)
    label_cur, label_other = 'NQ-Tables', 'OTT-QA'

else:
    _s1_other = list(read_jsonl(REPO_ROOT / 'results' / 'stage1' / 'nq_stage1.jsonl'))
    _precomp  = load_pickle(DATA_DIR / 'corpus_top5000_reranked_with_SentTrasf_title_head_desc.pkl')
    _s2_other = []
    for idx, it in enumerate(_s1_other):
        top = _precomp.get(str(idx), [])[:STAGE2_TOP_K]
        _s2_other.append({
            'qid'           : it['qid'],
            'gold_table_ids': it['gold_table_ids'],
            'retrieved'     : [{'rank': r+1, 'table_id': tid} for r, tid in enumerate(top)],
        })
    s1_metrics_other = evaluate_recall(_s1_other, EVAL_KS)
    s2_metrics_other = evaluate_recall(_s2_other, EVAL_KS)
    n_cur, n_other   = len(stage1_results), len(_s1_other)
    label_cur, label_other = 'OTT-QA', 'NQ-Tables'

# ── Build comparison table ────────────────────────────────────────────────────
rows = []
for k in EVAL_KS:
    rows.append({
        'k'                                  : k,
        f'{label_cur} Stage1 (SPLADE)'       : f"{s1_metrics_cur[k]:.4f}",
        f'{label_cur} Stage2 (Dense)'        : f"{s2_metrics_cur[k]:.4f}",
        f'{label_cur} Improvement'           : pct_improvement(s1_metrics_cur[k], s2_metrics_cur[k]),
        f'{label_other} Stage1 (SPLADE)'     : f"{s1_metrics_other[k]:.4f}",
        f'{label_other} Stage2 (Dense)'      : f"{s2_metrics_other[k]:.4f}",
        f'{label_other} Improvement'         : pct_improvement(s1_metrics_other[k], s2_metrics_other[k]),
    })

df = pd.DataFrame(rows).set_index('k')

print(f'\n{"=" * 76}')
print(f'  Stage 1 → Stage 2 Recall@k  |  {label_cur} (n={n_cur})  /  {label_other} (n={n_other})')
print(f'  Mode: {"precomputed" if USE_PRECOMPUTED else "computed"}')
print(f'  Improvement = relative gain over Stage 1  [ (S2−S1)/S1 × 100 ]')
print(f'{"=" * 76}')
display(df)

# ── Persist summary to CSV ────────────────────────────────────────────────────
summary_path = RESULTS_DIR / 'recall_summary.csv'
new_rows = []
for dataset, stage, metrics, n in [
    (label_cur,   'stage1', s1_metrics_cur,   n_cur),
    (label_cur,   'stage2', s2_metrics_cur,   n_cur),
    (label_other, 'stage1', s1_metrics_other, n_other),
    (label_other, 'stage2', s2_metrics_other, n_other),
]:
    row = {'dataset': dataset, 'stage': stage,
           'mode': 'precomputed' if USE_PRECOMPUTED else 'computed',
           'num_queries': n}
    for k in EVAL_KS:
        row[f'recall@{k}'] = metrics[k]
    new_rows.append(row)

new_df = pd.DataFrame(new_rows)
if summary_path.exists():
    combined = pd.concat([pd.read_csv(summary_path), new_df], ignore_index=True)
else:
    combined = new_df
combined.to_csv(summary_path, index=False)
print(f'\nSummary saved → {summary_path}')